# Reopen a Clerk case: Atera human kidney

Scratch notebook (not part of the docs) for browsing a Clerk case saved in `Clerk_Atera_Kidney_Landscape-Clustergram.ipynb`: its questions, answers and evidence, the cell types Clerk proposed, and the labels on the Clustergram (yours and Clerk's).

`dega.viz.Clerk(clerk_files=...)` reopens the case on fresh widgets showing the same data. **No Claude needed**:

- the panel shows the whole conversation; its genes, clusters, cited views and screenshots change the Landscape and Clustergram;
- the Clustergram shows the case's cell types (`cell_type` columns) and the marker genes behind them (`info` rows);
- each question's view is a pin on the Landscape (**PINS**): hover one for its question, click it to return to that view;
- a new question continues the case (that needs Claude), and new saves go beside it.

The last section stages the case for Hugging Face, where the gallery page `gallery/gallery_clerk_atera_kidney.md` and `Clerk(clerk_files=<folder URL>)` read it.

In [ ]:
import os

# Use this checkout's JS build, served locally (restart the kernel after a rebuild).
os.environ["CELLDEGA_LOCAL_ESM"] = "serve"

In [ ]:
import json
from pathlib import Path
import shutil
from urllib.request import urlopen

import celldega as dega
import numpy as np

SAMPLE = "Atera_v1_WTA_Kidney_non_diseased_FFPE"
DATA_REVISION = "3a46ab4db67821eef3b8ff8d1c3704fb7b3bb48f"
BASE_URL = f"https://raw.githubusercontent.com/broadinstitute/Celldega_{SAMPLE}/{DATA_REVISION}"

# The saved case: a conversation from Save (.json), a report folder from Report, or
# a folder of them (clerk_files/atera_kidney opens the most recent).
CASE = Path("clerk_files/atera_kidney/conversations/conversation.json")

# The clustered Matrix the case was made with (written by the main kidney notebook).
cwd = Path.cwd().resolve()
repo = next(p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file())
matrix_store = repo / "notebooks/data/atera_data" / SAMPLE / "kidney_leiden_marker_matrix.zarr"
if not matrix_store.exists():
    raise FileNotFoundError(
        f"{matrix_store} not found: run Clerk_Atera_Kidney_Landscape-Clustergram.ipynb once to compute it"
    )
matrix = dega.clust.Matrix.read_zarr(matrix_store)

with urlopen(f"{BASE_URL}/micron_to_image_transform.csv") as response:
    transform = np.loadtxt(response)

## The case, on its data

The widgets are made first and passed to Clerk, so the case's views become pins on the Landscape and its labels go on the Clustergram.

In [ ]:
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera kidney — default clusters", height=600, max_tiles_to_view=25,
)
cgm = dega.viz.Clustergram(
    matrix=matrix, width=400, height=500, rank_dim=10,
    manual_col_cat="cell_type", manual_row_cat="info",
)
clerk = dega.viz.Clerk(clerk_files=CASE, landscape=landscape, clustergram=cgm)
print("Opened", clerk._case["path"], "saved", clerk._case.get("saved"))
print(len(landscape.pins), "pins;", len(clerk.findings), "proposals")

In [ ]:
linked = dega.viz.spatial_clustergram(landscape, cgm, width="500px", clerk=clerk)

In [ ]:
linked

## Cell types

One row per cluster labeled on the Clustergram or proposed by Clerk: the label and who set it, and Clerk's latest proposal.

In [ ]:
clerk.cell_types()

Every proposal, with its markers and the question it answered:

In [ ]:
clerk.findings_df()

Each question's views as Snapshot cards (e.g. `clerk.snapshots.card("Q1 Landscape")`):

In [ ]:
clerk.snapshots

## Stage the case for Hugging Face

The gallery page `gallery/gallery_clerk_atera_kidney.md` reads the case folder from
Hugging Face (`clerk/atera_kidney/` in `broadinstitute/Celldega_Supporting_Data`), with
no kernel, and Python can reopen it with `Clerk(clerk_files=<folder URL>)`. This
writes an upload-ready copy of the case to `data/hf_upload/clerk/atera_kidney/` (git
ignores `data` folders):

- the conversation, its analysis notebooks and Enrichr cache, with paths on this
  computer replaced: the Matrix by its URL on Hugging Face, the analyses by their
  path in the case folder;
- the clustered Matrix, so the folder is self-contained;
- a short README.

Rerun it after saving a new version of the case, then upload the folder.

In [ ]:
HF_REPO = "broadinstitute/Celldega_Supporting_Data"
HF_PATH = "clerk/atera_kidney"
HF_CASE = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{HF_PATH}"
HF_MATRIX = f"{HF_CASE}/kidney_leiden_markers.zarr"

case_dir = CASE.parent.parent if CASE.parent.name == "conversations" else CASE.parent
stage = repo / "data/hf_upload" / HF_PATH
if stage.exists():
    shutil.rmtree(stage)
(stage / "conversations").mkdir(parents=True)


def portable(text):
    """Replace paths on this computer with where they are in the shared case."""
    for local in {str(matrix_store), str(matrix_store.resolve())}:
        text = text.replace(local, HF_MATRIX)
    return text.replace(f"{case_dir.as_posix()}/analyses/", "analyses/")


(stage / "conversations/conversation.json").write_text(portable(CASE.read_text(encoding="utf-8")))
for folder in ("analyses", "lookup_cache"):
    for path in sorted((case_dir / folder).glob("*")):
        (stage / folder).mkdir(exist_ok=True)
        (stage / folder / path.name).write_text(portable(path.read_text(encoding="utf-8")))
shutil.copytree(matrix_store, stage / "kidney_leiden_markers.zarr",
                ignore=shutil.ignore_patterns(".DS_Store"))
(stage / "README.md").write_text(f"""# Clerk case: Atera kidney cell types

A saved conversation with Clerk, Celldega's AI tool for interpreting its
visualizations, about the default Leiden clusters of the public 10x Genomics Atera
WTA FFPE human non-diseased kidney dataset, with the cell-type labels it put on the
Clustergram.

- `conversations/conversation.json`: the conversation, its findings and labels.
- `kidney_leiden_markers.zarr`: the clustered marker Matrix behind the Clustergram.
- `analyses/`: analysis notebooks Clerk drafted during the conversation.
- `lookup_cache/`: the Enrichr results Clerk looked up.

Reopen it in Python (the tissue streams from the DegaFiles on GitHub):

```python
import celldega as dega

mat = dega.clust.Matrix.read_zarr("{HF_MATRIX}")
cgm = dega.viz.Clustergram(matrix=mat)
clerk = dega.viz.Clerk(clerk_files="{HF_CASE}/", clustergram=cgm)
```
""")

# Nothing in the shared copy should point at this computer.
leftover = [str(p.relative_to(stage)) for p in stage.rglob("*")
            if p.is_file() and str(Path.home()) in p.read_text(errors="ignore")]
assert not leftover, f"local paths left in {leftover}"
size = sum(p.stat().st_size for p in stage.rglob("*") if p.is_file())
print(f"Staged {stage.relative_to(repo)} ({size / 1e6:.0f} MB). Upload it with:")
print(f"hf upload {HF_REPO} {stage.relative_to(repo)} {HF_PATH} --repo-type dataset "
      '--commit-message "Clerk case: Atera kidney cell types"')

After uploading, reopen the case from Hugging Face to check it (the Clustergram reads
the uploaded Matrix):

In [ ]:
online_cgm = dega.viz.Clustergram(
    matrix=dega.clust.Matrix.read_zarr(HF_MATRIX), width=400, height=500, rank_dim=10,
    manual_col_cat="cell_type", manual_row_cat="info",
)
online = dega.viz.Clerk(clerk_files=f"{HF_CASE}/", clustergram=online_cgm)
online.cell_types().head()